# 🎯 [MNT06_M02] CAD 파일 메쉬 정제, 슬라이싱 데이터 파싱 및 제작 규격 전처리 모듈 구현
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MNT06/mnt06_m02_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장비정비·수리부속 (Level 3) |
| **미션 ID** | MNT06_M02 |
| **미션 주제** | CAD 파일 메쉬 정제, 슬라이싱 데이터 파싱 및 제작 규격 전처리 모듈 구현 |
| **소요 시간** | 약 2시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
수리부속의 폐기율이 높으면 정비비용과 재고 부담이 커지고, 조달이 늦어지는 동안 장비는 가동하지 못한다. 폐기가 잦은 부품 가운데 야전에서 3D 프린팅으로 대체 제작할 수 있는 부품을 가려내면 폐기 손실과 조달 대기를 함께 줄일 수 있다. 본 프로젝트는 부품 폐기율의 산정·예측을 대체 제작 타당성 판단과 연결하는 것을 목표로, 교육용 가상 부품 자료(CAD 요약 치수, 금속·폴리머 소재 물성, 사용 하중, 폐기율, 조달 소요일)를 코드로 만들어 분석한다. 데이터 정의와 메쉬·슬라이싱 전처리, 형상 기반 Baseline과 탐색적 분석, 인장강도·피로 수명 예측과 군 규격 부합 신뢰도 점수, 대시보드와 설명가능성, 정비기술관 승인(HITL) 시뮬레이터, 50종 일괄 평가 스트레스 시험, 파괴 시험 대비 AAR 재보정과 효과 산출, 재학습을 포함한 통합 도구까지 단계별로 구현한다. 최종 목표는 폐기 손실이 큰 부품 중 신뢰성이 확인된 대체 제작 후보를 근거와 함께 제시하고, 실제 장착 여부는 정비기술관이 결정하도록 지원하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📁 2. 실습 산출물(Deliverables) 디렉터리 구조 및 안내

> **💡 [학습 안내] 산출물 디렉터리 아키텍처**  
> 본 과제는 단순 일회성 노트북 실행에 그치지 않고, **실제 군수 정비 야전에 즉시 도입 가능한 "완성형 국방 3D 프린팅 대체제작 타당성 및 부품 폐기율 예측 도구"를 완성하는 실무 엔지니어링 과정**입니다. 각 미션의 **산출물(Deliverables)**에 명시된 디렉터리 경로는 실무 표준 소프트웨어 아키텍처에 해당합니다:

| 디렉토리 | 군 실무 소프트웨어 아키텍처 분류 | 주요 보관 산출물 | 관련 미션 |
| :--- | :--- | :--- | :---: |
| **`docs/`** | **국방 도메인 및 데이터 규격서** | 군수 도메인 분석서 및 데이터 딕셔너리(`docs/mnt06_logistics_domain_spec.md`), 표준 운영 절차 기술 매뉴얼(`docs/mnt06_operations_guide.md`) | 미션 1, 10 |
| **`src/etl/`** | **CAD 메쉬 정제 및 전처리 모듈** | 3D CAD 메쉬 정제, 슬라이싱 데이터 파싱 파이프라인(`src/etl/mnt06_data_cleansing.py`) | 미션 2 |
| **`notebooks/`** | **탐색적 데이터 분석(EDA) & Baseline** | 기하 형상 기반 1차 제작 가능 여부 판별 Baseline(`notebooks/mnt06_eda_analysis.ipynb`) | 미션 3 |
| **`src/features/`** | **FEA 연계 피처 엔지니어링** | 유한요소해석 연계 응력 계산 및 적층 공정 피처 엔지니어링(`src/features/mnt06_feature_engineering.py`) | 미션 4 |
| **`src/models/`** | **AI 예측 & 군 규격 신뢰도 엔진** | 인장강도·피로내구도 예측 및 MIL-STD 규격 부합 신뢰도 엔진(`src/models/mnt06_advanced_model.py`) | 미션 5 |
| **`src/explainability/`** | **XAI 설명가능성 모듈** | 소요 시간·비용 분석 및 신뢰도 평가 결과 XAI 원인 설명기(`src/explainability/mnt06_xai_explainer.py`) | 미션 6 |
| **`src/optimization/`** | **HITL 승인 절차 시뮬레이터** | 정비기술관 HITL 검토·승인 큐 및 야전 부품 실장착 시뮬레이터(`src/optimization/mnt06_simulator.py`) | 미션 7 |
| **`src/dashboard/`** | **스트레스 시험 & 관제 대시보드** | 복잡 형상 부품 50종 일괄 신뢰성 평가 시뮬레이션 대시보드(`src/dashboard/mnt06_app.py`) | 미션 8 |
| **`reports/`** | **검증 및 AAR 리포트** | 사후검토(AAR) 파괴 시험 결과 오차 분석 및 ROI 보고서(`reports/mnt06_roi_analysis.md`) | 미션 9 |
| **`pipeline/`** | **MLOps 자동 재학습** | 실물 피드백 기반 자동 재학습 및 E2E 통합 평가 파이프라인(`pipeline/mnt06_retrain.py`) | 미션 10 |

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 02. CAD 파일 메쉬 정제, 슬라이싱 데이터 파싱 및 제작 규격 전처리 모듈 구현
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

CAD 메쉬 정제 및 슬라이싱 데이터 파싱 전처리 모듈을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **CAD 메쉬 정제와 슬라이싱 데이터 파싱, 부품 이력 레코드 정제를 수행하는 제작 규격 전처리 모듈을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **삼각형 메쉬의 NaN 정점·퇴화 삼각형·중복 삼각형을 제거하고 뒤집힌 법선을 바로잡아 부피가 기준과 일치하는지 확인하고 유효성을 점검하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/etl/mnt06_data_cleansing.py 및 정제 품질 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 특수 예외(비계획 정비, 긴급 불출 등) 데이터 보존의 타당성

#### 미션 PASS 조건

- 정량 기준: 데이터 정제 규칙 10종 적용, 데이터 정합성 검증 통과율 100%


- 진행 기준: ETL 파이프라인 구현 및 정제된 마스터 데이터셋 구축


- 검증 방법: 전후 데이터 통계량(기술통계) 비교 및 무결성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
# ============================================================================
# [미션 02 산출물 파일 연계]: src/etl/mnt06_data_cleansing.py
# (선택) 로컬 모듈 파일로 추출 저장 시: %%writefile src/etl/mnt06_data_cleansing.py
# ============================================================================
"""부품 이력 레코드(표기·단위·중복·결측 오류), CAD 삼각형 메쉬(NaN 정점·퇴화·중복·뒤집힌 법선), 슬라이싱(G-code류) 출력 파일의 불량 라인이 섞인 가상 원천 자료에 정제 규칙 12종을 적용하고, 긴급 제작 예외 레코드는 보존하면서 정합성 검증 통과율과 정제 전후 통계를 확인한다."""
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer

# ---------------------------------------------------------------------------
# [교육용가상자료] 야전 3D 프린팅 대체 제작 후보 부품 생성기 (실제 부품·재고·규격 값 아님)
# 폐기율이 높은 수리부속 중 3D 프린팅으로 대체 제작할 수 있는지 판단하기 위한 가상 부품 자료를 만든다.
# 소재 물성·치수·하중은 교육용으로 단순화한 값이며 실제 시험 성적서나 군 규격 값과 무관하다.
# ---------------------------------------------------------------------------
MAT = {  # 소재: (공정, 기준 인장강도 MPa, 밀도 g/cm3, 소재 단가 만원/kg, 출력비 만원/h, 최소 벽두께 mm, 출력 속도 cm3/h)
    "PA12": ("polymer", 45, 1.0, 8, 3, 0.8, 18), "PETG": ("polymer", 50, 1.27, 5, 2, 0.8, 15),
    "CF-PA": ("polymer", 70, 1.2, 14, 3, 1.0, 12), "AlSi10Mg": ("metal", 350, 2.67, 30, 25, 0.5, 4),
    "SS316L": ("metal", 520, 7.9, 28, 25, 0.6, 3.5), "Ti64": ("metal", 900, 4.4, 90, 30, 0.6, 3)}
CATS = ["bracket", "housing", "clip", "gear", "duct", "handle"]
CAT_SCRAP = {"bracket": 0.04, "housing": 0.05, "clip": 0.12, "gear": 0.09, "duct": 0.08, "handle": 0.10}   # 부품군별 기본 월 폐기율
BUILD_MM = {"polymer": 300, "metal": 200}                                                                 # 가상 출력 가능 최대 치수


def make_parts(n=300, seed=0):
    """가상 부품 n종: 형상(부피·벽두께·돌출 비율), 소재, 사용 하중, 폐기율과 조달 정보를 만든다."""
    rng = np.random.default_rng(seed)
    mat = rng.choice(list(MAT), n, p=[0.2, 0.15, 0.1, 0.2, 0.2, 0.15])
    proc = np.array([MAT[m][0] for m in mat])
    vol = np.clip(rng.lognormal(np.log(30), 0.9, n), 2, 800)
    aspect = rng.lognormal(0.3, 0.4, n)
    max_dim = 10 * vol ** (1 / 3) * aspect ** (1 / 3) * rng.uniform(1.0, 1.6, n)
    wall = np.clip(rng.lognormal(np.log(2.0), 0.6, n), 0.3, 8)
    ovh = rng.beta(2, 5, n)
    is_poly = proc == "polymer"
    df = pd.DataFrame({
        "part_id": [f"P{i:03d}" for i in range(n)], "cat": rng.choice(CATS, n, p=[0.2, 0.15, 0.2, 0.1, 0.15, 0.2]), "material": mat, "process": proc,
        "volume_cm3": vol, "max_dim_mm": max_dim, "area_cm2": 6 * vol ** (2 / 3) * rng.uniform(1.0, 2.2, n), "min_wall_mm": wall, "overhang": ovh,
        "n_holes": rng.poisson(2.0, n), "thread": rng.random(n) < 0.25, "orient_deg": rng.uniform(0, 90, n),
        "layer_mm": np.where(is_poly, rng.choice([0.1, 0.2, 0.3], n), rng.choice([0.03, 0.05], n)),
        "infill": np.where(is_poly, rng.choice([40, 60, 80, 100], n, p=[0.15, 0.3, 0.3, 0.25]), 100),
        "load_N": np.clip(rng.lognormal(np.where(is_poly, np.log(230), np.log(2300)), 0.7, n), 10, 20000), "temp_C": rng.uniform(-20, 90, n),
        "crit": rng.choice([1, 2, 3], n, p=[0.35, 0.4, 0.25]), "cert": rng.choice([0, 1], n, p=[0.45, 0.55]),   # 소재 인증서 보유
        "import_part": rng.random(n) < 0.4})
    df["scrap_rate"] = np.clip(df["cat"].map(CAT_SCRAP) * rng.lognormal(0, 0.35, n) * (1 + 0.5 * (df["load_N"] > 800)) * (1 + 0.3 * (df["temp_C"] > 60)), 0.005, 0.5)
    df["lead_days"] = np.clip(rng.lognormal(np.log(25), 0.5, n) * np.where(df["import_part"], 2.2, 1.0), 5, 300)
    # ---- 숨은 진실값(실제 시험 결과에 해당): 모델은 위 입력으로 이것을 예측한다 ----
    eff = np.where(is_poly, 0.55 + 0.45 * df["infill"] / 100, 1 - 0.3 * ovh * 0.2)
    aniso = np.where(is_poly, 1 - 0.3 * np.sin(np.radians(df["orient_deg"])) ** 2, 1 - 0.08 * np.sin(np.radians(df["orient_deg"])))
    heat = np.where(is_poly, 1 - 0.35 * np.clip((df["temp_C"] - 40) / 80, 0, 1), 0.98)
    base = np.array([MAT[m][1] for m in mat])
    df["strength_mpa"] = base * eff * aniso * heat * rng.lognormal(0, 0.06, n)
    cross = df["min_wall_mm"] * np.sqrt(df["max_dim_mm"]) * 1.0                                  # 하중 지지 단면적 mm2(단순화)
    df["stress_mpa"] = df["load_N"] / cross
    df["safety"] = df["strength_mpa"] / df["stress_mpa"]
    ratio = np.clip(df["stress_mpa"] / df["strength_mpa"], 0, 1.5)
    df["log_cycles"] = np.clip(7.5 - 6.0 * ratio + rng.normal(0, 0.25, n), 2, 7.5)               # 피로 수명 log10(회)
    support = 1 + 0.5 * ovh
    df["build_h"] = vol * support / np.array([MAT[m][6] for m in mat]) + 0.5
    mass_kg = vol * np.array([MAT[m][2] for m in mat]) * support / 1000
    df["print_cost"] = mass_kg * np.array([MAT[m][3] for m in mat]) + df["build_h"] * np.array([MAT[m][4] for m in mat]) + 3
    df["buy_cost"] = df["print_cost"] * rng.lognormal(0.25, 0.45, n) + 5                      # 구매 단가(만원, 가상)
    wall_ok = df["min_wall_mm"] >= np.array([MAT[m][5] for m in mat]) * rng.uniform(0.95, 1.25, n)
    dim_ok = df["max_dim_mm"] <= np.where(is_poly, BUILD_MM["polymer"], BUILD_MM["metal"])
    warp = (~is_poly) & (df["max_dim_mm"] > 140) & (df["overhang"] > 0.3)                     # 금속 대형 오버행 변형
    thr_bad = df["thread"] & (df["min_wall_mm"] < 1.5) & is_poly                              # 얇은 나사부 파손
    df["printable"] = wall_ok & dim_ok & (ovh < 0.5) & ~warp & ~thr_bad
    risk = np.clip(1 - df["safety"] / 3, 0, 1)
    df["conform"] = np.clip(100 * (0.45 * np.minimum(df["safety"] / 3, 1) + 0.25 * np.clip((df["log_cycles"] - 3) / 4.5, 0, 1)
                                   + 0.15 * (1 - np.clip(ovh * 1.2, 0, 1)) + 0.15 * df["cert"]) - 8 * risk * df["crit"] / 3 + rng.normal(0, 2, n), 0, 100)
    df["demand_month"] = rng.poisson(np.where(df["cat"].isin(["clip", "handle"]), 40, 15)) + 1               # 월 소요수량(가상)
    df["scrap_cost"] = df["scrap_rate"] * df["demand_month"] * df["buy_cost"]                              # 월 폐기 손실액(만원, 가상)
    return df

rng = np.random.default_rng(2)
applied = []


def rule(name, n):
    applied.append((name, int(n)))


# ---------------- 1) 부품 이력 레코드 ----------------
P = make_parts(200, seed=5)
raw = P[["part_id", "material", "max_dim_mm", "min_wall_mm", "volume_cm3", "scrap_rate", "load_N"]].copy()
raw["unit"] = "mm"
raw["test_date"] = [f"2026-{rng.integers(1, 10):02d}-{rng.integers(1, 28):02d}" for _ in range(len(raw))]
raw["emergency"] = False
idx = rng.choice(len(raw), 80, replace=False)
a, b, c, d, e, f, g = np.split(idx[:77], 7)
raw.loc[a, "material"] = rng.choice(["pa12", "PA-12", "ss316l", "SS 316L", "ti64"], len(a))          # 소재 표기 불일치
raw.loc[b, ["max_dim_mm", "min_wall_mm"]] = raw.loc[b, ["max_dim_mm", "min_wall_mm"]].values / 25.4   # inch 값
raw.loc[b, "unit"] = "inch"
raw.loc[c, "scrap_rate"] = (raw.loc[c, "scrap_rate"] * 100).round(1)                                 # % 단위 혼재
raw.loc[d, "test_date"] = [x.replace("-", "/") for x in raw.loc[d, "test_date"]]                    # 날짜 형식 혼재
raw.loc[e, "min_wall_mm"] = rng.choice([0.0, -1.2], len(e))                                          # 불가능 값
raw.loc[f, ["load_N", "volume_cm3"]] = np.nan                                                        # 결측
raw.loc[g, "emergency"] = True; raw.loc[g, "load_N"] = raw.loc[g, "load_N"] * 12                    # 긴급 제작 요청(예외: 과대 하중 기록, 보존)
raw = pd.concat([raw, raw.sample(10, random_state=1)], ignore_index=True)                           # 중복 레코드
print(f"[부품 이력] 원천 {len(raw)}건 (중복 10건, 오류 주입 {len(idx[:77])}건 포함)")
df = raw.copy()
rule("R1 중복 레코드 제거", df.duplicated().sum()); df = df.drop_duplicates().reset_index(drop=True)
mat = df["material"].str.upper().str.replace(r"[\s-]", "", regex=True).map({"PA12": "PA12", "PETG": "PETG", "CFPA": "CF-PA", "ALSI10MG": "AlSi10Mg", "SS316L": "SS316L", "TI64": "Ti64"})
rule("R2 소재 코드 표기 통일", (mat != df["material"]).sum()); df["material"] = mat
inch = df["unit"] == "inch"
df.loc[inch, ["max_dim_mm", "min_wall_mm"]] *= 25.4; df["unit"] = "mm"; rule("R3 CAD 치수 단위 통일(inch -> mm)", inch.sum())
pct = df["scrap_rate"] > 1; df.loc[pct, "scrap_rate"] /= 100; rule("R4 폐기율 단위 통일(% -> 비율)", pct.sum())
t1 = pd.to_datetime(df["test_date"], format="%Y-%m-%d", errors="coerce")
t2 = pd.to_datetime(df["test_date"], format="%Y/%m/%d", errors="coerce"); rule("R5 시험일 형식 통일", t1.isna().sum()); df["test_date"] = t1.fillna(t2)
bad = df["min_wall_mm"] <= 0; df.loc[bad, "min_wall_mm"] = np.nan; rule("R6 불가능 벽두께 결측 처리", bad.sum())
miss = df[["load_N", "volume_cm3", "min_wall_mm"]].isna().sum().sum()
df[["load_N", "volume_cm3", "min_wall_mm"]] = KNNImputer(n_neighbors=5).fit_transform(df[["load_N", "volume_cm3", "min_wall_mm", "max_dim_mm"]])[:, :3]
rule("R7 결측 KNN 보정", miss)
q1, q3 = df["load_N"].quantile([0.25, 0.75]); out = df["load_N"] > q3 + 3 * (q3 - q1)
df["outlier_flag"] = out & ~df["emergency"]
rule("R12 긴급 제작 예외 보존·일반 이상값 표시", (out & df["emergency"]).sum())

# ---------------- 2) CAD 메쉬 ----------------
def box_tris(a_, b_, c_, k=4):
    dims, tris = (a_, b_, c_), []
    for ax in range(3):
        u, v = [i for i in range(3) if i != ax]
        for sgn in (0, 1):
            for i in range(k):
                for j in range(k):
                    def Pt(ii, jj):
                        p = np.zeros(3); p[ax] = dims[ax] * sgn; p[u] = dims[u] * ii / k; p[v] = dims[v] * jj / k; return p
                    p00, p10, p11, p01 = Pt(i, j), Pt(i + 1, j), Pt(i + 1, j + 1), Pt(i, j + 1)
                    flip = (-1 if ax == 1 else 1) != (1 if sgn else -1)
                    for t in ([p00, p10, p11], [p00, p11, p01]):
                        tris.append(t[::-1] if flip else t)
    return np.array(tris)


def volume(T):
    return float(np.einsum("ij,ij->i", T[:, 0], np.cross(T[:, 1], T[:, 2])).sum() / 6)


T0 = box_tris(40.0, 30.0, 20.0)                                   # 정상 메쉬(가상 브래킷 블랭크, 부피 24 cm3)
nan_extra = T0[rng.choice(len(T0), 3, replace=False)].copy(); nan_extra[:, 0] = np.nan          # NaN 정점을 가진 불량 삼각형
deg_extra = T0[rng.choice(len(T0), 8, replace=False)].copy(); deg_extra[:, 1] = deg_extra[:, 0]  # 퇴화(면적 0) 삼각형
dup_extra = T0[rng.choice(len(T0), 6, replace=False)].copy()                                     # 중복 삼각형
m = np.concatenate([T0, nan_extra, deg_extra, dup_extra])
fl = rng.choice(len(T0), 10, replace=False); m[fl] = m[fl][:, ::-1]                              # 법선 뒤집힘(원본 중 10개)
vol_raw = volume(np.nan_to_num(m))
n_raw = len(m)
nan_t = np.isnan(m).any(axis=(1, 2)); rule("R8 NaN 정점 삼각형 제거", nan_t.sum()); m = m[~nan_t]
area = 0.5 * np.linalg.norm(np.cross(m[:, 1] - m[:, 0], m[:, 2] - m[:, 0]), axis=1)
degen = area < 1e-9
keys = [tuple(sorted(map(tuple, np.round(t, 4)))) for t in m]
dup = np.array([k in set(keys[:i]) for i, k in enumerate(keys)])
rule("R9 퇴화·중복 삼각형 제거", (degen | dup).sum()); m = m[~(degen | dup)]
cen = m.mean(axis=(0, 1)); nrm = np.cross(m[:, 1] - m[:, 0], m[:, 2] - m[:, 0]); inward = np.einsum("ij,ij->i", nrm, m.mean(axis=1) - cen) < 0
m[inward] = m[inward][:, ::-1]; rule("R10 법선 방향 통일(안쪽 향함 뒤집기)", inward.sum())
vol_clean = volume(m)

# ---------------- 3) 슬라이싱 출력 파싱 ----------------
lines = ["; layer_height=0.2"]
for L in range(30):
    lines.append(f";LAYER:{L}")
    for s in range(12):
        lines.append(f"G1 X{rng.uniform(0, 40):.2f} Y{rng.uniform(0, 30):.2f} E{rng.uniform(0.01, 0.2):.3f} F{rng.choice([1200, 1800, 2400])}")
for i in rng.choice(len(lines), 8, replace=False):
    if lines[i].startswith("G1"):
        lines[i] = lines[i].replace(" Y", " Yx") if i % 2 else lines[i].replace("F", "F-")        # 좌표 깨짐·음수 이송속도
pat = re.compile(r"^G1 X(-?\d+\.\d+) Y(-?\d+\.\d+) E(\d+\.\d+) F(\d+)$")
good, rej, layer = [], [], -1
for ln in lines:
    if ln.startswith(";LAYER:"):
        layer = int(ln.split(":")[1])
    elif ln.startswith("G1"):
        mt = pat.match(ln)
        (good if mt else rej).append((layer, *map(float, mt.groups())) if mt else ln)
G = pd.DataFrame(good, columns=["layer", "x", "y", "e", "f"])
rule("R11 슬라이싱 불량 라인 격리(수동 확인 목록)", len(rej))

# ---------------- 4) 정합성 검증 ----------------
chk = {"중복 레코드 0건": not df.duplicated().any(), "소재 코드 허용값": df["material"].isin(list(MAT)).all(),
       "치수 단위 mm": (df["unit"] == "mm").all(), "폐기율 0~1": df["scrap_rate"].between(0, 1).all(),
       "시험일 변환": df["test_date"].notna().all(), "벽두께 0 초과": (df["min_wall_mm"] > 0).all(),
       "결측 0건": df[["load_N", "volume_cm3", "min_wall_mm"]].notna().all().all(),
       f"긴급 제작 {int(df['emergency'].sum())}건 보존": int(df["emergency"].sum()) == len(g),
       "메쉬 NaN 0건": not np.isnan(m).any(), "메쉬 부피 오차 1% 이내": abs(vol_clean - 24000) / 24000 < 0.01,
       "슬라이싱 이송속도 양수": bool((G["f"] > 0).all())}
rate = np.mean(list(chk.values()))
print("\n[적용 규칙과 처리 건수]")
applied.sort(key=lambda r_: int(re.match(r"R(\d+)", r_[0]).group(1)))
for nme, n in applied:
    print(f" {nme:42s} {n:4d}건")
print(f"\n정합성 검증 {sum(chk.values())}/{len(chk)} 통과, 통과율 {rate:.0%} (목표 100%: {'달성' if rate == 1 else '미달'}), 규칙 {len(applied)}종 (목표 10종 이상: {'달성' if len(applied) >= 10 else '미달'})")
for k, v in chk.items():
    if not v:
        print(" 실패:", k)
print(f"정제 후 부품 레코드 {len(df)}건, 일반 이상값 표시 {int(df['outlier_flag'].sum())}건, 긴급 제작 예외 {int(df['emergency'].sum())}건 유지")
print(f"메쉬 삼각형 {n_raw} -> {len(m)}개, 부피 {vol_raw / 1000:.2f} -> {vol_clean / 1000:.2f} cm3(기준 24.00)")
print(f"슬라이싱: {int(G['layer'].nunique())}층, 유효 {len(G)}줄, 격리 {len(rej)}줄, 총 압출량 {G['e'].sum():.1f}, 예상 이송 시간 {(np.hypot(G['x'].diff(), G['y'].diff()).fillna(0) / G['f'] * 60).sum():.2f}분")

num = lambda x: pd.to_numeric(x, errors="coerce")
cmp = pd.DataFrame({"정제 전 평균": [num(raw[c_]).mean() for c_ in ["max_dim_mm", "min_wall_mm", "scrap_rate", "load_N"]],
                    "정제 후 평균": [df[c_].mean() for c_ in ["max_dim_mm", "min_wall_mm", "scrap_rate", "load_N"]],
                    "정제 전 최소": [num(raw[c_]).min() for c_ in ["max_dim_mm", "min_wall_mm", "scrap_rate", "load_N"]],
                    "정제 후 최소": [df[c_].min() for c_ in ["max_dim_mm", "min_wall_mm", "scrap_rate", "load_N"]]},
                   index=["최대치수mm", "벽두께mm", "폐기율", "하중N"]).round(3)
print("\n[정제 전후 기술통계]"); print(cmp.to_string())

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(num(raw["scrap_rate"]).dropna(), bins=30, alpha=0.6, label="raw"); ax[0].hist(df["scrap_rate"], bins=30, alpha=0.6, label="cleaned")
ax[0].set_title("Scrap rate before/after unit fix"); ax[0].legend()
ax[1].bar([n_.split()[0] for n_, _ in applied], [n_ for _, n_ in applied]); ax[1].set_title("Records changed by rule")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
[부품 이력] 원천 210건 (중복 10건, 오류 주입 77건 포함)

[적용 규칙과 처리 건수]
 R1 중복 레코드 제거                                 10건
 R2 소재 코드 표기 통일                               11건
 R3 CAD 치수 단위 통일(inch -> mm)                  11건
 R4 폐기율 단위 통일(% -> 비율)                        11건
 R5 시험일 형식 통일                                 11건
 R6 불가능 벽두께 결측 처리                             11건
 R7 결측 KNN 보정                                 33건
 R8 NaN 정점 삼각형 제거                              3건
 R9 퇴화·중복 삼각형 제거                              14건
 R10 법선 방향 통일(안쪽 향함 뒤집기)                      10건
 R11 슬라이싱 불량 라인 격리(수동 확인 목록)                   7건
 R12 긴급 제작 예외 보존·일반 이상값 표시                     5건

정합성 검증 11/11 통과, 통과율 100% (목표 100%: 달성), 규칙 12종 (목표 10종 이상: 달성)
정제 후 부품 레코드 200건, 일반 이상값 표시 3건, 긴급 제작 예외 11건 유지
메쉬 삼각형 209 -> 192개, 부피 21.00 -> 24.00 cm3(기준 24.00)
슬라이싱: 30층, 유효 353줄, 격리 7줄, 총 압출량 39.0, 예상 이송 시간 238.36분

[정제 전후 기술통계]
         정제 전 평균   정제 후 평균  정제 전 최소  정제 후 최소
최대치수mm    43.270    46.097    0.817   18.448
벽두께mm      2.114     2.407   -1.200    0.405
폐기율        0.663     0.117    0.029    0.029
하중N     2785.396  2864.990   50.301   50.301
```
</details>
